In [1]:
import pandas as pd                                      
import numpy as np                                        
import matplotlib    

In [2]:
import matplotlib                                          # matplotlib: base plotting library
matplotlib.use("Agg")

In [3]:
import matplotlib.pyplot as plt                          
import seaborn as sns                                    
import joblib                                              
from sklearn.model_selection import train_test_split      
from sklearn.preprocessing import StandardScaler           
from sklearn.linear_model import LogisticRegression        
from sklearn.tree import DecisionTreeClassifier           
from sklearn.ensemble import RandomForestClassifier       
from sklearn.metrics import accuracy_score                
from sklearn.metrics import classification_report         
from sklearn.metrics import confusion_matrix     

In [4]:
df = pd.read_csv("amazon_products.csv")                    
print("Shape (rows, columns):", df.shape)                 
print(df.head())                                          
print(df.info()) 

Shape (rows, columns): (1500, 11)
  product_id category  actual_price  discount_percentage  discounted_price  \
0    B100000    Books         655.0                   45             360.0   
1    B100001   Beauty         520.0                   17             432.0   
2    B100002  Fashion        1338.0                   62             508.0   
3    B100003   Beauty        1852.0                   54             852.0   
4    B100004   Beauty         989.0                   34             653.0   

   is_prime  delivery_days  return_rate  review_sentiment  rating_count  \
0         1              2          7.0              0.35         290.0   
1         1              3         12.3              0.56          16.0   
2         1              2          9.2              0.54         145.0   
3         0              4         12.5              0.06         107.0   
4         0              3          7.4              0.41         234.0   

   rating  
0     4.0  
1     3.5  
2     4.0 

In [5]:
#: EXPLORE (EDA = Exploratory Data Analysis) ----------
print("\nMissing values per column:\n", df.isnull().sum())  # count empty cells in each column
print("\nSummary statistics:\n", df.describe())   


Missing values per column:
 product_id              0
category                0
actual_price            0
discount_percentage     0
discounted_price        0
is_prime                0
delivery_days           0
return_rate            60
review_sentiment        0
rating_count           40
rating                  0
dtype: int64

Summary statistics:
        actual_price  discount_percentage  discounted_price     is_prime  \
count   1500.000000          1500.000000       1500.000000  1500.000000   
mean    2275.217333            38.380000       1405.767333     0.612000   
std     2726.852385            19.054597       1860.276800     0.487457   
min      121.000000             5.000000         59.000000     0.000000   
25%      683.000000            22.000000        392.500000     0.000000   
50%     1304.500000            39.000000        745.500000     1.000000   
75%     2576.500000            55.000000       1561.250000     1.000000   
max    23141.000000            70.000000      1897

In [6]:
df["return_rate"] = df["return_rate"].fillna(df["return_rate"].median())     
df["rating_count"] = df["rating_count"].fillna(df["rating_count"].median())   
df = df.drop_duplicates()                                 
print("\nMissing after cleaning:", df.isnull().sum().sum()) 



Missing after cleaning: 0


In [7]:

df["good_product"] = (df["rating"] >= 4.0).astype(int)     
print("\nClass balance:\n", df["good_product"].value_counts())  



Class balance:
 good_product
0    1261
1     239
Name: count, dtype: int64


In [9]:
plt.figure(figsize=(6, 4))                                 
sns.countplot(x="good_product", data=df)                  
plt.title("Good (1) vs Not Good (0) Products")            
plt.savefig("chart_class_balance.png", bbox_inches="tight")  
plt.close()   

In [11]:
plt.figure(figsize=(6, 4))                             
sns.boxplot(x="category", y="rating", data=df)             
plt.xticks(rotation=45)                                
plt.title("Rating by Category")                         
plt.savefig("chart_rating_by_category.png", bbox_inches="tight")  
plt.close()  

In [12]:
df["price_saved"] = df["actual_price"] - df["discounted_price"]  
df = pd.get_dummies(df, columns=["category"], drop_first=True)   

In [13]:
X = df.drop(columns=["product_id", "rating", "good_product"])  
y = df["good_product"]                                           


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                                                 
    test_size=0.2,                                         
    random_state=42,                                       
    stratify=y)   

In [16]:
scaler = StandardScaler()                                  # creates the scaler (makes mean=0, std=1)
X_train_scaled = scaler.fit_transform(X_train)             # learn scale from TRAIN data and apply it
X_test_scaled = scaler.transform(X_test)  

In [17]:
models = {                                                 # dictionary holding the 3 models so we can loop over them
    "Logistic Regression": LogisticRegression(max_iter=1000),                     # simple linear model; max_iter gives it enough steps to learn
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),        # tree of if/else questions; max_depth=5 prevents overfitting
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),   # 200 trees voting together = usually most accurate
}

In [18]:
results = {}                                               # empty dictionary to store each model's accuracy

for name, model in models.items():                         # loop through each model one by one
    model.fit(X_train_scaled, y_train)                     # TRAINING: model learns patterns from train data
    preds = model.predict(X_test_scaled)                   # PREDICTING: model guesses answers for unseen test data
    acc = accuracy_score(y_test, preds)                    # compare guesses with real answers -> accuracy
    results[name] = acc                                    # store accuracy
    print(f"\n===== {name} =====")                         # print model name heading
    print("Accuracy:", round(acc, 4))                      # print accuracy rounded to 4 decimals
    print(classification_report(y_test, preds))            # precision/recall/f1 for each class
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))  # [[correct 0, wrong], [wrong, correct 1]]

# ---------- STEP 11: COMPARE MODELS ----------
print("\nModel comparison:", results)                      # show all accuracies together
best_name = max(results, key=results.get)                  # find the model name with the highest accuracy
best_model = models[best_name]                             # get that model object
print("Best model:", best_name)                            # print the winner

plt.figure(figsize=(6, 4))                                 # new chart
plt.bar(results.keys(), results.values())                  # bar chart of accuracies
plt.ylim(0, 1)                                             # y-axis from 0 to 1 (0% to 100%)
plt.xticks(rotation=15)                                    # slight rotation for readable labels
plt.title("Model Accuracy Comparison")                     # chart title
plt.savefig("chart_model_comparison.png", bbox_inches="tight")  # save image
plt.close()                                                # close chart

# ---------- STEP 12: FEATURE IMPORTANCE (which columns matter most?) ----------
rf = models["Random Forest"]                               # take the Random Forest model
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)  # importance score per column, highest first
print("\nTop features:\n", importance.head(8))             # print the 8 most important features

plt.figure(figsize=(8, 5))                                 # new chart
importance.head(8).sort_values().plot(kind="barh")         # horizontal bar chart of top 8 features
plt.title("Top Features Driving Product Rating")           # chart title
plt.savefig("chart_feature_importance.png", bbox_inches="tight")  # save image
plt.close()                                                # close chart

# ---------- STEP 13: SAVE THE MODEL ----------
joblib.dump(best_model, "best_model.pkl")                  # save best model to a file so we can reuse it without retraining
joblib.dump(scaler, "scaler.pkl")                          # save the scaler too (new data must be scaled the same way)
joblib.dump(list(X.columns), "feature_columns.pkl")        # save column order (model expects the exact same order)

# ---------- STEP 14: PREDICT FOR A NEW PRODUCT ----------
new_product = {                                            # a made-up new product (a Prime electronics item)
    "actual_price": 5000, "discount_percentage": 30, "discounted_price": 3500,
    "is_prime": 1, "delivery_days": 2, "return_rate": 3.0,
    "review_sentiment": 0.5, "rating_count": 800, "price_saved": 1500,
}
new_df = pd.DataFrame([new_product])                       # convert dictionary into a one-row table
new_df = new_df.reindex(columns=X.columns, fill_value=0)   # add missing columns (category dummies) as 0 and fix the order. All category columns = 0 means Electronics, because drop_first removed it
new_scaled = scaler.transform(new_df)                      # scale using the saved scaler
pred = best_model.predict(new_scaled)[0]                   # predict class: 0 or 1
prob = best_model.predict_proba(new_scaled)[0][1]          # probability (0 to 1) that product is good
print("\nNew product prediction:", "GOOD product" if pred == 1 else "NOT good product")  # human-readable answer
print("Probability of being good:", round(prob, 3))        # confidence of the model



===== Logistic Regression =====
Accuracy: 0.9133
              precision    recall  f1-score   support

           0       0.92      0.98      0.95       252
           1       0.82      0.58      0.68        48

    accuracy                           0.91       300
   macro avg       0.87      0.78      0.82       300
weighted avg       0.91      0.91      0.91       300

Confusion Matrix:
 [[246   6]
 [ 20  28]]

===== Decision Tree =====
Accuracy: 0.9067
              precision    recall  f1-score   support

           0       0.92      0.97      0.95       252
           1       0.78      0.58      0.67        48

    accuracy                           0.91       300
   macro avg       0.85      0.78      0.81       300
weighted avg       0.90      0.91      0.90       300

Confusion Matrix:
 [[244   8]
 [ 20  28]]

===== Random Forest =====
Accuracy: 0.9
              precision    recall  f1-score   support

           0       0.91      0.98      0.94       252
           1      